In [1]:
import os
import json
import pandas as pd
import traceback

In [165]:
if "HUGGINGFACEHUB_API_TOKEN" in os.environ:
    del os.environ["HUGGINGFACEHUB_API_TOKEN"]

In [24]:
from dotenv import load_dotenv
load_dotenv()

True

In [164]:
import os
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Setup your endpoint and chat model (as you already did)
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct", # Cleaned up repo id if using standard HF
    task="text-generation",
    temperature=0.5,
    max_new_tokens=512
)
chat = ChatHuggingFace(llm=llm)

# 2. Define an output parser to extract clean string text from the model responses
output_parser = StrOutputParser()

# ----------------------------------------------------
# CHAIN 1: Generate a company name based on a product
# ----------------------------------------------------
prompt1 = ChatPromptTemplate.from_template(
    "What is a good, catchy name for a company that makes {product}?"
)
# Construct the first chain
chain_one = prompt1 | chat | output_parser

# ----------------------------------------------------
# CHAIN 2: Generate a tagline based on the company name
# ----------------------------------------------------
prompt2 = ChatPromptTemplate.from_template(
    "Write a catchy 3-word slogan or tagline for the following company: {company_name}"
)
# Construct the second chain
chain_two = prompt2 | chat | output_parser

print(output_parser)

# ----------------------------------------------------
# THE SEQUENTIAL CHAIN: Link them together
# ----------------------------------------------------
# We use a lambda function to map the output of chain_one into the input key of chain_two
sequential_chain = chain_one | (lambda company: {"company_name": company}) | chain_two

# 4. Run the complete pipeline
result = sequential_chain.invoke({"product": "eco-friendly running shoes"})
print(result)


ValueError: You must provide an api_key to work with auto API or log in with `hf auth login`.

In [72]:
import os
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Setup your endpoint and chat model (as you already did)
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct", # Cleaned up repo id if using standard HF
    task="text-generation",
    temperature=0.5,
    max_new_tokens=512
)
chat = ChatHuggingFace(llm=llm)

In [124]:
RESPONSE_JSON = {
    "1": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "2": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
    "3": {
        "mcq": "multiple choice question",
        "options": {
            "a": "choice here",
            "b": "choice here",
            "c": "choice here",
            "d": "choice here",
        },
        "correct": "correct answer",
    },
}

In [137]:
TEMPLATE="""
Text{text}
You are an expert MCQ maker. Given the above text, it is your job to \
create a quiz of {number} multiple choice questions for {subject} students in {tone} tone.
Make sute the questions are not repeated and check all the questions to be confirmed the text as well.
Make sure to format your response like RESPONSE_JSON below and use it as a guide. \
Ensure to make {number} MCQs
### RESPONSE_JSON
{response_json}

"""
chain_one = ChatPromptTemplate.from_template(TEMPLATE) | chat | output_parser

In [138]:
TEMPLATE2="""
You are an expert english grammarian and writer. Given a Multiple Choice Quiz for {subject} students.\
You need to evaluate the complexity of the question and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. 
if the quiz is not at per with the cognitive and analytical abilities of the students,\
update the quiz questions which needs to be changed and change the tone such that it perfectly fits the student abilities
Quiz_MCQs:
{quiz}

Check from an expert English Writer of the above quiz:
"""
chain_two = ChatPromptTemplate.from_template(TEMPLATE2) | chat | output_parser

In [139]:
from langchain_core.runnables import RunnablePassthrough

# Assuming 'quiz_chain' and 'review_chain' are already defined earlier in your code
# (e.g., quiz_chain = prompt1 | chat | output_parser)

generate_evaluate_chain = (
    # 1. Run quiz_chain and append its output to the running state dictionary as 'quiz'
    RunnablePassthrough.assign(quiz=chain_one)
    
    # 2. Run review_chain using the updated state (which now includes 'quiz' and 'subject') 
    #    and append its output as 'review'
    | RunnablePassthrough.assign(review=chain_two)
)

In [140]:
file_path = r'/Users/roshni/Documents/Hasib/Personal Project/First Ai Project/data.txt'

with open(file_path,'r') as file:
    TEXT = file.read()

# print(TEXT)

In [141]:
# Conveting the python dictionary into a JSON-formatted string
json.dumps(RESPONSE_JSON)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}, "3": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct": "correct answer"}}'

In [158]:
NUMBER = 5
SUBJECT ='globalization in india'
TONE = 'simple'

In [153]:
pip install langchain_community

Note: you may need to restart the kernel to use updated packages.


In [159]:
from langchain_community.callbacks import get_openai_callback

# Wrap your invoke call inside the context manager
with get_openai_callback() as cb:
    result = generate_evaluate_chain.invoke({
        'text': TEXT,
        'number': NUMBER,
        'subject': SUBJECT,
        'tone': TONE,
        'response_json': json.dumps(RESPONSE_JSON)
    })
    
    # Print the token breakdown
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")

# Your existing code continues down here
quiz = result.get('quiz')

Total Tokens: 4429
Prompt Tokens: 3552
Completion Tokens: 877
Total Cost (USD): $0.0


In [160]:
quiz = json.loads(quiz)
print(quiz)

{'1': {'mcq': "What percentage of the world's GDP did India account for at the end of the Mughal era?", 'options': {'a': '25.9%', 'b': '32.9%', 'c': '17.9%', 'd': '42.9%'}, 'correct': 'b'}, '2': {'mcq': "What was the value of India's international trade in 2003-04?", 'options': {'a': '12.50 billion', 'b': '63,0801 billion', 'c': '30.80 billion', 'd': '56.80 billion'}, 'correct': 'b'}, '3': {'mcq': "Which of the following countries were India's major trading partners?", 'options': {'a': 'China, the US, the UAE, the UK, Japan, and the EU', 'b': 'China, the UK, Japan, and the EU', 'c': 'The US, the UAE, the UK, and the EU', 'd': 'The UK, Japan, and the EU'}, 'correct': 'a'}, '4': {'mcq': "What was the average annual rate of India's economic growth from 2002-2012?", 'options': {'a': '3.5%', 'b': '5.5%', 'c': '7.7%', 'd': '9.5%'}, 'correct': 'c'}, '5': {'mcq': "What percentage of the world's remittances did India claim in 2007?", 'options': {'a': '8%', 'b': '10%', 'c': '12%', 'd': '15%'}, '

In [161]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value["mcq"]
    options = " | ".join(
        [
            f"{option}: {option_value}"
            for option, option_value in value["options"].items()
            ]
        )
    correct = value["correct"]
    quiz_table_data.append({"MCQ": mcq, "Choices": options, "Correct": correct})
    print(quiz_table_data)

[{'MCQ': "What percentage of the world's GDP did India account for at the end of the Mughal era?", 'Choices': 'a: 25.9% | b: 32.9% | c: 17.9% | d: 42.9%', 'Correct': 'b'}]
[{'MCQ': "What percentage of the world's GDP did India account for at the end of the Mughal era?", 'Choices': 'a: 25.9% | b: 32.9% | c: 17.9% | d: 42.9%', 'Correct': 'b'}, {'MCQ': "What was the value of India's international trade in 2003-04?", 'Choices': 'a: 12.50 billion | b: 63,0801 billion | c: 30.80 billion | d: 56.80 billion', 'Correct': 'b'}]
[{'MCQ': "What percentage of the world's GDP did India account for at the end of the Mughal era?", 'Choices': 'a: 25.9% | b: 32.9% | c: 17.9% | d: 42.9%', 'Correct': 'b'}, {'MCQ': "What was the value of India's international trade in 2003-04?", 'Choices': 'a: 12.50 billion | b: 63,0801 billion | c: 30.80 billion | d: 56.80 billion', 'Correct': 'b'}, {'MCQ': "Which of the following countries were India's major trading partners?", 'Choices': 'a: China, the US, the UAE, the 

In [162]:
pd.DataFrame(quiz_table_data)

,MCQ,Choices,Correct
0,What percentage of the world's GDP did India a...,a: 25.9% | b: 32.9% | c: 17.9% | d: 42.9%,b
1,What was the value of India's international tr...,"a: 12.50 billion | b: 63,0801 billion | c: 30....",b
2,Which of the following countries were India's ...,"a: China, the US, the UAE, the UK, Japan, and ...",a
3,What was the average annual rate of India's ec...,a: 3.5% | b: 5.5% | c: 7.7% | d: 9.5%,c
4,What percentage of the world's remittances did...,a: 8% | b: 10% | c: 12% | d: 15%,c
